# Docker - Wprowadzenie

Prosty przewodnik po Dockerze w kontekście uruchamiania PostgreSQL dla materiałów szkoleniowych.

---

## 1. Co to Docker i dlaczego go używamy?

### Problem bez Dockera:

```
Chcesz uruchomić PostgreSQL na swoim komputerze:

❌ Musisz zainstalować PostgreSQL (wersja, dependencies, PATH...)
❌ Konfiguracja (użytkownik, hasło, baza, port...)
❌ "Na moim komputerze działa" - na innych może nie
❌ Konflikt wersji (masz już PostgreSQL 14, potrzebujesz 15)
❌ Trudne odinstalowanie (zostają resztki w systemie)
❌ Każdy musi robić to samo (Windows, Mac, Linux - różnie)
```

### Rozwiązanie - Docker:

```bash
docker-compose up -d
```

**I to wszystko!**

```
✅ PostgreSQL uruchomiony w 5 sekund
✅ Izolowany od systemu (nie konfliktuje z innymi wersjami)
✅ Działa identycznie na Windows, Mac, Linux
✅ Łatwe usunięcie (docker-compose down -v)
✅ Wszyscy mają to samo środowisko
```

### Analogia: Kontener transportowy

```
🚢 Kontener transportowy:
├─ Standardowy rozmiar i interfejs
├─ Można przewozić statkiem, pociągiem, ciężarówką
├─ Zawartość izolowana (nie miesza się z innymi)
└─ Łatwy transport i składowanie

🐳 Kontener Docker:
├─ Standardowy format (Docker)
├─ Można uruchomić na Windows, Mac, Linux, Cloud
├─ Aplikacja izolowana (własne środowisko)
└─ Łatwe uruchamianie i usuwanie
```

## 2. Docker vs Tradycyjna instalacja

### Tradycyjna instalacja PostgreSQL:

```bash
# Ubuntu/Debian:
sudo apt-get update
sudo apt-get install postgresql-15 postgresql-contrib
sudo systemctl start postgresql
sudo -u postgres psql
CREATE USER myuser WITH PASSWORD 'mypass';
CREATE DATABASE mydb OWNER myuser;
# ... konfiguracja pg_hba.conf, postgresql.conf ...

# Windows: pobierz installer, klikaj Next...
# Mac: brew install postgresql, ale która wersja?
```

**Problemy:**
- Długa procedura
- Różna na każdym OS
- Zmiany w systemie (service, użytkownicy, porty)

### Docker:

```bash
docker-compose up -d
```

**Zalety:**
- Jedna komenda, każdy OS
- Zero zmian w systemie
- Łatwe usunięcie: `docker-compose down -v`

### Docker vs Virtual Machine (VM):

```
Virtual Machine:              Docker Container:
┌───────────────┐             ┌───────────────┐
│ PostgreSQL    │             │ PostgreSQL    │
├───────────────┤             │ (tylko proces)│
│ Cały OS Linux │             ├───────────────┤
│ (~1-2 GB RAM) │             │ Docker Engine │
├───────────────┤             ├───────────────┤
│ Hypervisor    │             │ Host OS       │
├───────────────┤             └───────────────┘
│ Host OS       │             
└───────────────┘             Rozmiar: ~100 MB
                              RAM: ~50-100 MB
Rozmiar: ~5 GB                Start: sekundy
RAM: ~1-2 GB
Start: minuty
```

**Docker = lekki, szybki, praktyczny**

## 3. Podstawowe pojęcia

### Image (obraz)

**Image = przepis / szablon**

```
postgres:15-alpine = oficjalny image PostgreSQL 15

- Zawiera: PostgreSQL, wszystkie dependencies, konfigurację
- Tylko do odczytu (template)
- Pobierany z Docker Hub (repozytorium obrazów)
```

```bash
# Zobacz lokalne images:
docker images

# REPOSITORY          TAG          SIZE
# postgres            15-alpine    ~200MB
```

### Container (kontener)

**Container = działająca instancja image**

```
Image:     postgres:15-alpine (template)
            ↓ docker run
Container: produkty_postgres (running process)
```

```bash
# Zobacz działające kontenery:
docker ps

# CONTAINER ID   IMAGE               STATUS    PORTS
# abc123         postgres:15-alpine  Up 5min   5432->5432
```

**Analogia:**
- **Image** = przepis na ciasto
- **Container** = upieczone ciasto (możesz mieć wiele ciast z jednego przepisu)

### Volume (wolumin)

**Volume = trwałe przechowywanie danych**

```
Problem: Dane w kontenerze są tymczasowe
├─ docker stop → dane pozostają
└─ docker rm → DANE ZNIKAJĄ!

Rozwiązanie: Volume
├─ Dane przechowywane POZA kontenerem
├─ Przetrwają restart, usunięcie kontenera
└─ Można współdzielić między kontenerami
```

```yaml
volumes:
  - postgres_data:/var/lib/postgresql/data
#   ↑ nazwa         ↑ ścieżka w kontenerze
```

**Bez volume:**
```bash
docker-compose down
docker-compose up -d
# WSZYSTKIE DANE ZNIKNĘŁY!
```

**Z volume:**
```bash
docker-compose down
docker-compose up -d
# Dane nadal tam są!
```

## 4. Docker Compose - co to i po co?

### Problem: Długie komendy docker run

```bash
# Uruchomienie PostgreSQL bez Docker Compose:
docker run -d \
  --name produkty_postgres \
  -e POSTGRES_USER=postgres \
  -e POSTGRES_PASSWORD=postgres \
  -e POSTGRES_DB=produkty_db \
  -p 5432:5432 \
  -v postgres_data:/var/lib/postgresql/data \
  --health-cmd="pg_isready -U postgres" \
  --health-interval=10s \
  --health-timeout=5s \
  --health-retries=5 \
  postgres:15-alpine

# Problemy:
❌ Długa, skomplikowana komenda
❌ Trudna do zapamiętania
❌ Trzeba powtarzać przy każdym uruchomieniu
❌ Ryzyko literówki
```

### Rozwiązanie: Docker Compose

**Docker Compose = deklaratywna konfiguracja w YAML**

Zamiast długiej komendy, tworzysz plik `docker-compose.yml`:

```yaml
version: '3.8'

services:
  postgres:
    image: postgres:15-alpine
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: postgres
      POSTGRES_DB: produkty_db
    ports:
      - "5432:5432"
    volumes:
      - postgres_data:/var/lib/postgresql/data

volumes:
  postgres_data:
```

I teraz:

```bash
docker-compose up -d
# Gotowe!
```

**Zalety:**
- Czytelna konfiguracja
- Łatwa do edycji (zmień port, hasło, wersję...)
- Można commitować do git (współdzielić z zespołem)
- Jedna komenda zamiast długiego bash script

## 5. Nasz docker-compose.yml - omówienie

### Plik docker-compose.yml:

```yaml
version: '3.8'

services:
  postgres:
    image: postgres:15-alpine
    container_name: produkty_postgres
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: postgres
      POSTGRES_DB: produkty_db
    ports:
      - "5432:5432"
    volumes:
      - postgres_data:/var/lib/postgresql/data
      - ./data/init.sql:/docker-entrypoint-initdb.d/init.sql
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 10s
      timeout: 5s
      retries: 5

volumes:
  postgres_data:
```

### Omówienie każdej sekcji:

#### 1. `version: '3.8'`

```yaml
version: '3.8'
# Wersja składni docker-compose
# 3.8 = aktualna, stabilna wersja
```

#### 2. `services:`

```yaml
services:
  postgres:  # Nazwa service (możesz użyć innej)
```

Sekcja `services` definiuje kontenery do uruchomienia.

#### 3. `image:`

```yaml
image: postgres:15-alpine
#      ↑         ↑    ↑
#      |         |    └─ wariant (alpine = mniejszy)
#      |         └────── wersja (15)
#      └────────────────── nazwa obrazu
```

**Warianty:**
- `postgres:15` - pełny obraz (~350 MB)
- `postgres:15-alpine` - lekki obraz (~200 MB) **← używamy**

#### 4. `container_name:`

```yaml
container_name: produkty_postgres
# Nazwa kontenera (łatwiej go znaleźć w docker ps)
```

#### 5. `environment:`

```yaml
environment:
  POSTGRES_USER: postgres       # Użytkownik bazy
  POSTGRES_PASSWORD: postgres   # Hasło (zmień w produkcji!)
  POSTGRES_DB: produkty_db      # Nazwa bazy danych
```

Te zmienne są używane przy pierwszym uruchomieniu (inicjalizacja bazy).

#### 6. `ports:`

```yaml
ports:
  - "5432:5432"
#    ↑    ↑
#    |    └─ port w kontenerze (PostgreSQL domyślny)
#    └────── port na hoście (twój komputer)
```

**Możesz zmienić port hosta:**
```yaml
ports:
  - "5433:5432"  # Host port 5433, kontener 5432
# Teraz: psql -h localhost -p 5433
```

#### 7. `volumes:`

```yaml
volumes:
  # Named volume (zarządzany przez Docker)
  - postgres_data:/var/lib/postgresql/data
  
  # Bind mount (mapowanie pliku z hosta)
  - ./data/init.sql:/docker-entrypoint-initdb.d/init.sql
```

**Named volume:**
- Dane bazy (tabele, rekordy)
- Przetrwają restart kontenera
- Usunięcie: `docker-compose down -v`

**Bind mount:**
- `./data/init.sql` (plik na dysku) → `/docker-entrypoint-initdb.d/init.sql` (w kontenerze)
- PostgreSQL automatycznie wykonuje pliki `.sql` z tego katalogu przy pierwszym starcie
- Inicjalizacja bazy danymi

#### 8. `healthcheck:`

```yaml
healthcheck:
  test: ["CMD-SHELL", "pg_isready -U postgres"]
  # Komenda sprawdzająca czy baza jest gotowa
  
  interval: 10s  # Sprawdzaj co 10 sekund
  timeout: 5s    # Timeout po 5 sekundach
  retries: 5     # 5 prób zanim uznamy za failed
```

**Po co?**
- PostgreSQL może wystartować (kontener up), ale jeszcze nie być gotowy do połączeń
- Health check czeka aż baza naprawdę działa
- Przydatne gdy masz wiele kontenerów (app czeka aż db będzie healthy)

#### 9. `volumes:` (definicja)

```yaml
volumes:
  postgres_data:
    # Deklaracja named volume
    # Docker utworzy i zarządza tym woluminem
```

## 6. Podstawowe komendy Docker Compose

### Start i Stop:

```bash
# Uruchom wszystko (w tle)
docker-compose up -d
#                  ↑
#                  └─ detached (w tle, nie blokuje terminala)

# Uruchom z wyświetlaniem logów (foreground)
docker-compose up
# Ctrl+C aby zatrzymać

# Stop kontenerów (kontenery i volumes pozostają)
docker-compose stop

# Stop i usuń kontenery (volumes pozostają)
docker-compose down

# Stop, usuń kontenery I volumes (UTRATA DANYCH!)
docker-compose down -v

# Restart
docker-compose restart
```

### Logi:

```bash
# Zobacz logi
docker-compose logs

# Follow logs (live, na żywo)
docker-compose logs -f
#                    ↑
#                    └─ follow (jak tail -f)

# Ostatnie 50 linii
docker-compose logs --tail=50

# Logi z timestampem
docker-compose logs -t

# Logi konkretnego service
docker-compose logs postgres
```

### Status:

```bash
# Status kontenerów
docker-compose ps

# Output:
#        Name                   State          Ports
# produkty_postgres   Up (healthy)   0.0.0.0:5432->5432/tcp
#                     ↑
#                     └─ healthy = health check passed

# Top (CPU, memory usage)
docker-compose top
```

### Execute komend w kontenerze:

```bash
# Połącz się z bazą (psql)
docker-compose exec postgres psql -U postgres -d produkty_db
#               ↑        ↑     ↑
#               |        |     └─ komenda do wykonania w kontenerze
#               |        └─────── nazwa service z docker-compose.yml
#               └──────────────── execute w działającym kontenerze

# Wejdź do shell kontenera
docker-compose exec postgres sh
# (alpine używa sh zamiast bash)

# Wykonaj SQL z hosta
docker-compose exec -T postgres psql -U postgres -d produkty_db -c "SELECT * FROM produkty;"
#                    ↑
#                    └─ -T = disable TTY (dla skryptów)
```

### Volumes:

```bash
# Lista volumes
docker volume ls

# Informacje o volume
docker volume inspect data_sources_postgres_data

# Usuń volume (UWAGA: UTRATA DANYCH!)
docker volume rm data_sources_postgres_data

# Backup volume (przykład)
docker run --rm \
  -v data_sources_postgres_data:/data \
  -v $(pwd):/backup \
  alpine tar czf /backup/postgres_backup.tar.gz /data
```

### Przydatne aliasy:

```bash
# Dodaj do ~/.bashrc lub ~/.zshrc:
alias dcu='docker-compose up -d'
alias dcd='docker-compose down'
alias dcl='docker-compose logs -f'
alias dcp='docker-compose ps'
alias dcr='docker-compose restart'
```

## 7. Workflow - krok po kroku

### Pierwsze uruchomienie:

```bash
# 1. Przejdź do katalogu z docker-compose.yml
cd 00_extras/data_sources

# 2. Sprawdź plik docker-compose.yml
cat docker-compose.yml

# 3. Uruchom
docker-compose up -d

# Output:
# Creating network "data_sources_default" with the default driver
# Creating volume "data_sources_postgres_data" with default driver
# Pulling postgres (postgres:15-alpine)...
# Creating produkty_postgres ... done

# 4. Sprawdź status
docker-compose ps

# 5. Sprawdź logi
docker-compose logs
# Poszukaj: "database system is ready to accept connections"

# 6. Połącz się z bazą
docker-compose exec postgres psql -U postgres -d produkty_db

# W psql:
\dt          -- Lista tabel (powinno być: produkty)
SELECT * FROM produkty LIMIT 5;
\q           -- Wyjdź
```

### Codzienne użycie:

```bash
# Start (jeśli zatrzymany)
docker-compose up -d

# ... praca z bazą (Python notebooks) ...

# Stop (koniec dnia)
docker-compose stop
# Dane są bezpieczne w volume!

# Następnego dnia:
docker-compose start
# Wszystkie dane nadal tam są
```

### Reset bazy (start od nowa):

```bash
# UWAGA: To usunie wszystkie dane!

# 1. Stop i usuń kontenery + volumes
docker-compose down -v

# 2. Uruchom od nowa (baza będzie pusta, init.sql się wykona)
docker-compose up -d

# 3. Sprawdź czy dane z init.sql są wczytane
docker-compose exec postgres psql -U postgres -d produkty_db -c "SELECT COUNT(*) FROM produkty;"
#  count 
# -------
#     15
```

## 8. Troubleshooting

### Problem 1: Port już zajęty

```bash
# Error: Bind for 0.0.0.0:5432 failed: port is already allocated

# Przyczyna: Masz już PostgreSQL lub inny kontener na porcie 5432

# Sprawdź co używa portu:
sudo lsof -i :5432
# lub
sudo netstat -tulpn | grep 5432

# Rozwiązanie 1: Zatrzymaj konfliktujący service
sudo systemctl stop postgresql  # Jeśli to system PostgreSQL

# Rozwiązanie 2: Zmień port w docker-compose.yml
ports:
  - "5433:5432"  # Host: 5433, kontener: 5432

# Połączenie wtedy:
psql -h localhost -p 5433 -U postgres -d produkty_db
```

### Problem 2: Kontener nie startuje

```bash
# Sprawdź logi
docker-compose logs postgres

# Najczęstsze przyczyny:
# 1. Błąd w init.sql (syntax error w SQL)
# 2. Permissions na ./data/init.sql
# 3. Corrupted volume

# Rozwiązanie: Usuń volume i spróbuj ponownie
docker-compose down -v
docker-compose up -d
```

### Problem 3: Nie mogę się połączyć z bazą

```bash
# Error: psycopg2.OperationalError: connection refused

# Sprawdź status kontenera
docker-compose ps
# Powinno być: Up (healthy)

# Jeśli nie healthy:
docker-compose logs postgres

# Sprawdź czy port jest otwarty:
docker-compose port postgres 5432
# Powinno być: 0.0.0.0:5432

# Test połączenia:
docker-compose exec postgres pg_isready -U postgres
# Powinno być: accepting connections
```

### Problem 4: Wolumen się nie usuwa

```bash
# Error: volume is in use

# Sprawdź czy kontener nie działa:
docker-compose ps
docker ps -a  # Wszystkie kontenery (nawet zatrzymane)

# Usuń kontenery:
docker-compose down

# Teraz usuń volume:
docker volume rm data_sources_postgres_data

# Lub force:
docker-compose down -v
```

### Problem 5: Brakuje danych z init.sql

```bash
# init.sql wykonuje się TYLKO przy pierwszym starcie!
# Jeśli volume już istnieje, init.sql NIE jest wykonywany

# Rozwiązanie: Usuń volume i uruchom od nowa
docker-compose down -v
docker-compose up -d

# Sprawdź logi - powinno być:
docker-compose logs | grep init.sql
# .../docker-entrypoint-initdb.d/init.sql: running
```

## 9. Podsumowanie

### Co się nauczyłeś:

✅ **Co to Docker:**
- Konteneryzacja aplikacji
- Izolacja, reprodukowalność, przenośność
- Lżejszy niż VM

✅ **Podstawowe pojęcia:**
- **Image** = template (postgres:15-alpine)
- **Container** = running instance
- **Volume** = persistent storage

✅ **Docker Compose:**
- Deklaratywna konfiguracja w YAML
- Jedna komenda zamiast długich docker run
- Łatwe zarządzanie wieloma kontenerami

✅ **Podstawowe komendy:**
```bash
docker-compose up -d      # Start
docker-compose down       # Stop (dane pozostają)
docker-compose down -v    # Stop + usuń dane
docker-compose logs -f    # Logi (live)
docker-compose ps         # Status
docker-compose exec       # Execute w kontenerze
```

### Kluczowe wnioski:

```
Docker Compose = prosta konfiguracja
    ↓
docker-compose up -d = wszystko działa
    ↓
Volumes = dane są bezpieczne
    ↓
docker-compose down = łatwe sprzątanie
```

### Typowy workflow:

```bash
# Raz:
docker-compose up -d      # Pierwszy start (pobierze image, utworzy volume)

# Codziennie:
docker-compose start      # Uruchom (jeśli zatrzymany)
# ... praca ...
docker-compose stop       # Zatrzymaj (dane bezpieczne)

# Reset:
docker-compose down -v    # Usuń wszystko, zacznij od nowa
docker-compose up -d      # Fresh start
```

### Następne kroki:

- Przejdź do `06_postgresql.ipynb` i użyj bazy z Dockera
- Eksperymentuj z innymi obrazami (Redis, MongoDB, nginx...)
- Dla bardziej zaawansowanych setup: zobacz `00_extras/06_docker_deployment.ipynb`

---

**Docker = prosty, szybki, praktyczny! 🐳**